In [16]:
from src.retrievers.tf_idf import TFIDF
from src.retrievers.base_retriever import Retriever

docs = [{'text': ['the cat sat on the mat',
                  'the dog sat on the log',
                  'cats and dogs'],
         'document_id': ['d0', 'd1', 'd2']}]

class DevTFIDF(TFIDF):
    def __init__(self, documents):
        Retriever.__init__(self, documents)   # only builds self.index, skips the TFIDF stubs
        
        self.corpus_vocabulary = self.GetCorpusVocabulary()
        self.idf = self.GetInverseDocumentFrequencies()

        # Only run if we are in TFIDF class
        # if type(self) == TFIDF:
        self.tfidf_matrix = self.GetDocumentsTFIDFVectors()

t = DevTFIDF(docs) # instantiate the class, to access the methods

In [17]:
print(t.PreprocessText("The cat sat on the mat."))
print(t.GetQueryVocabulary("cat on the mat"))
print(t.GetCorpusVocabulary()) 


['the', 'cat', 'sat', 'on', 'the', 'mat.']
['mat', 'the', 'on', 'cat']
{'mat', 'on', 'sat', 'dog', 'cats', 'dogs', 'the', 'and', 'log', 'cat'}


In [18]:
from src.data_utils.document import Document

doc = Document(text="the cat sat on the mat", _id="d0")

print(t.GetDocumentVocabulary(doc))
print(t.GetDocumentTermCounts(doc))


{'mat', 'on', 'sat', 'the', 'cat'}
{'mat': 1, 'on': 1, 'sat': 1, 'the': 2, 'cat': 1}


In [19]:
print(t.GetDocumentFrequencies())
print(t.GetInverseDocumentFrequencies())

{'mat': 1, 'on': 2, 'sat': 2, 'the': 2, 'cat': 1, 'dog': 1, 'log': 1, 'and': 1, 'dogs': 1, 'cats': 1}
{'mat': np.float64(1.0986122886681098), 'on': np.float64(0.4054651081081644), 'sat': np.float64(0.4054651081081644), 'the': np.float64(0.4054651081081644), 'cat': np.float64(1.0986122886681098), 'dog': np.float64(1.0986122886681098), 'log': np.float64(1.0986122886681098), 'and': np.float64(1.0986122886681098), 'dogs': np.float64(1.0986122886681098), 'cats': np.float64(1.0986122886681098)}


### Queries

In [20]:
from src.data_utils.query import Query


class DevQuery(Query):
    def __init__(self, text: str, id: str, relevant_document_ids: list[str], label: str):
        
        self.text = text
        self.id = id
        self.relevant_document_ids = relevant_document_ids
        self.label = label


queries = [
    DevQuery(text='unrelated dogs',         id='q0', relevant_document_ids=['d0'], label='q0'),
    DevQuery(text='hello dogs',            id='q1', relevant_document_ids=['d1'], label='q1'),
    DevQuery(text='cats and dogs',          id='q2', relevant_document_ids=['d2'], label='q2'),
]

q0,q1,q2 = queries

In [21]:
q1.getQuery()

'hello dogs'

In [22]:
[t.GetQueryVocabulary(q.getQuery()) for q in queries]


[['unrelated', 'dogs'], ['hello', 'dogs'], ['and', 'dogs', 'cats']]

In [23]:
[(q.getQuery()) for q in queries]

['unrelated dogs', 'hello dogs', 'cats and dogs']

In [24]:
docs

[{'text': ['the cat sat on the mat',
   'the dog sat on the log',
   'cats and dogs'],
  'document_id': ['d0', 'd1', 'd2']}]

In [25]:
[t.QueryToVector([q.getQuery()]) for q in queries]


[<Compressed Sparse Row sparse matrix of dtype 'float32'
 	with 1 stored elements and shape (1, 10)>,
 <Compressed Sparse Row sparse matrix of dtype 'float32'
 	with 1 stored elements and shape (1, 10)>,
 <Compressed Sparse Row sparse matrix of dtype 'float32'
 	with 3 stored elements and shape (1, 10)>]

In [26]:
q1.getQuery()

'hello dogs'

In [27]:
[t.CalculateScores([q.getQuery()]) for q in queries]

[array([[0.       , 0.       , 1.0986123]], dtype=float32),
 array([[0.       , 0.       , 1.0986123]], dtype=float32),
 array([[0.      , 0.      , 3.295837]], dtype=float32)]

BM25


In [28]:
from src.retrievers.bm25 import BM25
from src.retrievers.base_retriever import Retriever

class DevBM25(BM25):
    def __init__(self, documents, k1=1.5, b=0.75):
        Retriever.__init__(self, documents)   
        self.k1 = k1
        self.b = b
        
        self.corpus_vocabulary = self.GetCorpusVocabulary()
        self.idf = self.GetInverseDocumentFrequencies()
        self.bm25_matrix = self.GetDocumentsBM25Vectors()


b_dev = DevBM25(docs)

In [29]:
print("Corpus Vocabulary", b_dev.GetCorpusVocabulary())
print("BM25 IDF Values", b_dev.GetInverseDocumentFrequencies())


doc_lens, avg_len = b_dev.GetDocumentLengths()
print("Document Lengths", {getattr(doc, 'id', getattr(doc, '_id', str(doc))): length for doc, length in doc_lens.items()})
print("Average Document Length", avg_len)


print("BM25 Matrix ", b_dev.bm25_matrix.shape)

Corpus Vocabulary {'mat', 'on', 'sat', 'dog', 'cats', 'dogs', 'the', 'and', 'log', 'cat'}
BM25 IDF Values {'mat': np.float64(0.9808292530117262), 'on': np.float64(0.47000362924573563), 'sat': np.float64(0.47000362924573563), 'the': np.float64(0.47000362924573563), 'cat': np.float64(0.9808292530117262), 'dog': np.float64(0.9808292530117262), 'log': np.float64(0.9808292530117262), 'and': np.float64(0.9808292530117262), 'dogs': np.float64(0.9808292530117262), 'cats': np.float64(0.9808292530117262)}
Document Lengths {'d0': 6, 'd1': 6, 'd2': 3}
Average Document Length 5.0
BM25 Matrix  (3, 10)


Query test

In [30]:

print("Query Vectors")
print([b_dev.QueryToVector([q.getQuery()]).toarray() for q in queries])


for q in queries:
    scores = b_dev.CalculateScores([q.getQuery()])
    print(f"Query '{q.id}' ('{q.getQuery()}')")
    print(scores)

Query Vectors
[array([[0., 0., 0., 0., 0., 1., 0., 0., 0., 0.]], dtype=float32), array([[0., 0., 0., 0., 0., 1., 0., 0., 0., 0.]], dtype=float32), array([[0., 0., 0., 0., 1., 1., 0., 1., 0., 0.]], dtype=float32)]
Query 'q0' ('unrelated dogs')
[[0.        0.        1.1961333]]
Query 'q1' ('hello dogs')
[[0.        0.        1.1961333]]
Query 'q2' ('cats and dogs')
[[0.     0.     3.5884]]


In [31]:

bm25_created= BM25(docs)

print("Scores Oficiais do BM25:")
query_texts = [q.getQuery() for q in queries]
print(bm25_created.CalculateScores(query_texts))

Scores Oficiais do BM25:
[[0.        0.        1.1961333]
 [0.        0.        1.1961333]
 [0.        0.        3.5884   ]]
